In [ ]:
# Tell Jupyter to share its event loop with PyQt
%gui

In [3]:
import os
from pprint import pprint
# print(os.environ)

In [6]:
import requests
import xml.etree.ElementTree as ET

# The exact text string to search
search_text = "Prolongación de Ramón y Cajal 5 Santa Cruz de Tenerife"
print(f"[INFO] Querying GRAFCAN API for: '{search_text}'...")

grafcan_api_url = "https://visor.grafcan.es/busquedas/toponimoxml/1/50/"
params = {'texto': search_text}

# 1. Send the HTTP request
response = requests.get(grafcan_api_url, params=params)

if response.status_code == 200:
    try:
        # 2. Parse the XML response text you provided
        root = ET.fromstring(response.text)

        # Find the first <row> element
        first_result = root.find('.//row')

        if first_result is not None:
            # Extract the raw text from the <x> and <y> tags and cast to floats
            lon = float(first_result.find('x').text)
            lat = float(first_result.find('y').text)
            nombre = first_result.find('nombre').text

            print(f"[SUCCESS] GRAFCAN Match Found: {nombre}")
            print(f"[INFO] Raw WGS 84 Extracted: Lon {lon}, Lat {lat}")

        else:
            print("[WARNING] GRAFCAN returned an empty XML sheet. No <row> found.")

    except ET.ParseError:
        print("[ERROR] Failed to parse the XML. Ensure the API is returning valid XML.")
else:
    print(f"[ERROR] HTTP Request failed with status code: {response.status_code}")



[INFO] Querying GRAFCAN API for: 'Prolongación de Ramón y Cajal 5 Santa Cruz de Tenerife'...
[SUCCESS] GRAFCAN Match Found: Prolongación Ramón y Cajal 5
[INFO] Raw WGS 84 Extracted: Lon -16.25368308, Lat 28.46562134


Transform coordinates using GDAL

In [8]:
from osgeo import osr
osr.UseExceptions()

lat = 28.46562134
lon = -16.25368308

# 1. Define the Source CRS (WGS84)
source_crs = osr.SpatialReference()
source_crs.ImportFromEPSG(4326)

# GDAL 3+ strictly follows EPSG rules (Lat/Lon).
# We force it to use standard GIS order (Lon/Lat) so it behaves predictably.
source_crs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)

# 2. Define the Target CRS (UTM 28N)
target_crs = osr.SpatialReference()
target_crs.ImportFromEPSG(32628)
target_crs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)

# 3. Build the transformation bridge
transform = osr.CoordinateTransformation(source_crs, target_crs)

# 4. Do the math (Passing X/Lon, Y/Lat)
# Note: TransformPoint returns a tuple of (X, Y, Z)
utm_x, utm_y, utm_z = transform.TransformPoint(lon, lat)

print(f"UTM X: {utm_x:.3f}")
print(f"UTM Y: {utm_y:.3f}")

UTM X: 377263.240
UTM Y: 3149423.470


Alternatively, transform coordinates using PyQGIS

In [4]:
# import sys
#
# from qgis.core import QgsApplication
#
# # Tell Python where your QGIS installation lives
# # (Change 'qgis-ltr' to just 'qgis' if your folder is named differently)
# QgsApplication.setPrefixPath(r"C:\OSGeo4W\apps\qgis-ltr", True)
#
# # QgsApplication.setPrefixPath(os.path.join(os.environ['OSGEO4W_ROOT'], r'apps\qgis-ltr'), True)
#
# # Initialize the QGIS application natively without a GUI (True)
# qgs = QgsApplication([], False)
# qgs.initQgis()
#
# print("[SUCCESS] QGIS Core Engine initialized!")

[SUCCESS] QGIS Core Engine initialized!


In [ ]:
# from qgis.core import (
#     QgsPointXY,
#     QgsCoordinateReferenceSystem,
#     QgsCoordinateTransform,
#     QgsProject
# )
#
# # 3. PyQGIS Reprojection Engine (WGS 84 -> UTM 28N)
# print("[INFO] Projecting to UTM Zone 28N (EPSG:32628)...")
#
# crs_wgs84 = QgsCoordinateReferenceSystem("EPSG:4326")
# crs_utm28n = QgsCoordinateReferenceSystem("EPSG:32628")
#
# # Create the math transformation context
# transform_context = QgsProject.instance().transformContext()
# transform = QgsCoordinateTransform(crs_wgs84, crs_utm28n, transform_context)
#
# # Apply the transformation (QgsPointXY expects Longitude first, then Latitude)
# pt_wgs = QgsPointXY(lon, lat)
# pt_utm = transform.transform(pt_wgs)
#
# # Save variables for the next cell
# utm_x = pt_utm.x()
# utm_y = pt_utm.y()
#
# print(f"[SUCCESS] Final UTM Coordinates -> X: {utm_x:.2f}, Y: {utm_y:.2f}")


In [9]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

shapefile_path = r"C:\dev\QGIS-testing\Modulo3\puntos_recogida.shp"

# 1. Open the shapefile (Just like st_read in R)
gdf = gpd.read_file(shapefile_path)


In [12]:
gdf.loc[gdf['id'] == 3, 'HORARIO'] = 'sin restricción horaria'
# Save the shapefile, overwriting the old one
gdf.to_file(shapefile_path)

print("[SUCCESS] Data corrected and shapefile overwritten!")

[SUCCESS] Data corrected and shapefile overwritten!


In [14]:
new_point = Point(utm_x, utm_y) # UTM 28N coordinates

new_data = gpd.GeoDataFrame({
    'id': [4],
    'RESIDUO': ['recogida selectiva de textil'],
    'UNIDADES': [1],
    'HORARIO': ['sin restricción horaria'],
    'CODIGO': ['SC017'],
    'T1_Field2': [nombre],
    'T1_Field3': ['https://ecoembesdudasreciclaje.es/contenedor-ropa-autorizado/'],
    'T1_Field4': ['SC017.jpg'],
    'UTMX': [utm_x],
    'UTMY': [utm_y],
    'geometry': [new_point]
}, crs="EPSG:32628") # Define the CRS (Just like st_as_sf)

# 3. Append the new row to the existing dataframe (Just like rbind in R)
updated_gdf = pd.concat([gdf, new_data], ignore_index=True)

# 4. Save it back to the shapefile (Just like st_write in R)
updated_gdf.to_file(shapefile_path)

print("[SUCCESS] Point added seamlessly without QGIS!")

[SUCCESS] Point added seamlessly without QGIS!


Verify extra data from the Excel file

In [15]:
import geopandas as gpd
import pandas as pd

# 1. Define your file paths
shapefile_path = r"C:\dev\QGIS-testing\Modulo3\puntos_recogida.shp"
csv_path = r"C:\dev\QGIS-testing\Modulo3\T1.csv"  # Ensure this is in your working directory

# 2. Load the Shapefile and the CSV
# (GeoPandas automatically handles the shapefile as a dataframe)
gdf = gpd.read_file(shapefile_path)
df = pd.read_csv(csv_path, encoding='utf-8')

# 3. Perform an Outer Merge with an Indicator
# 'outer' means it keeps all rows from both files, even if they don't match.
comparison_df = gdf.merge(df, on='CODIGO', how='outer', indicator=True)

# 4. Select only the relevant columns to compare them cleanly
# T1_Field2 is your shapefile direction, DIRECCION is the CSV direction
columns_to_show = ['id', 'CODIGO', 'T1_Field2', 'DIRECCION', '_merge']
print(comparison_df[columns_to_show])

# 5. (Optional) Create a specific check for records missing in the CSV
missing_in_csv = comparison_df[comparison_df['_merge'] == 'left_only']
if not missing_in_csv.empty:
    print("\n[ALERT] The following records are in the Shapefile but missing from the CSV:")
    print(missing_in_csv[['id', 'CODIGO']])

   id  CODIGO                                  T1_Field2  \
0   2  CT0615             Calle Rubens Marichal Lopez 70   
1   3   P0457  Calle Alcalde José Emilio García Gómez 5A   
2   1   R1296              Calle Veinticinco de Julio 45   
3   4   SC017               Prolongación Ramón y Cajal 5   

                                   DIRECCION     _merge  
0             Calle Rubens Marichal Lopez 70       both  
1  Calle Alcalde José Emilio García Gómez 5A       both  
2              Calle Veinticinco de Julio 45       both  
3                                        NaN  left_only  

[ALERT] The following records are in the Shapefile but missing from the CSV:
   id CODIGO
3   4  SC017


In [ ]:
# from qgis.core import (
#     QgsProject,
#     QgsFeature,
#     QgsGeometry,
#     QgsPointXY,
#     QgsVectorLayer,
#     QgsField,
#     QgsFeatureRequest
# )
# from PyQt5.QtCore import QVariant
#
# layer_name = 'puntos_recogida'
# layers = QgsProject.instance().mapLayersByName(layer_name)
#
# # 1. LAYER CREATION & 2. UNINDENTED MAIN PATH
# if not layers:
#     print(f"[INFO] Layer '{layer_name}' not found. Creating a new memory layer...")
#     # Create a point layer with Canarian UTM CRS
#     layer = QgsVectorLayer("Point?crs=epsg:32628", layer_name, "memory")
#
#     # Define the schema (columns)
#     provider = layer.dataProvider()
#     provider.addAttributes([
#         QgsField("id", QVariant.Int),
#         QgsField("RESIDUO", QVariant.String),
#         QgsField("UNIDADES", QVariant.Int),
#         QgsField("HORARIO", QVariant.String),
#         QgsField("CODIGO", QVariant.String),
#         QgsField("T1_Field2", QVariant.String),
#         QgsField("T1_Field3", QVariant.String),
#         QgsField("UTMX", QVariant.Double),
#         QgsField("UTMY", QVariant.Double)
#     ])
#     layer.updateFields()
#     QgsProject.instance().addMapLayer(layer)
# else:
#     layer = layers[0]


In [ ]:
# # At this point, 'layer' is guaranteed to exist. Main path continues unindented.
# target_id = 4
#
# # Assuming utm_x and utm_y were generated successfully in the previous Geocoding cell
# # utm_x = 377350.0  # (Fallback if previous cell wasn't run)
# # utm_y = 3149650.0 # (Fallback if previous cell wasn't run)
#
# layer.startEditing()
#
# # 3. OVERWRITE EXISTING ID
# # Query the layer for any feature where the attribute 'id' matches our target
# request = QgsFeatureRequest().setFilterExpression(f'"id" = {target_id}')
# existing_features = list(layer.getFeatures(request))
#
# if existing_features:
#     print(f"[INFO] Found existing feature(s) with id={target_id}. Deleting before overwrite...")
#     for existing_feat in existing_features:
#         # Delete the feature using QGIS's internal unique feature ID
#         layer.deleteFeature(existing_feat.id())
#
# # Create a blank feature using the layer's attribute table structure
# feat = QgsFeature(layer.fields())
#
# # Set the geometry and populate the attributes
# feat.setGeometry(QgsGeometry.fromPointXY(QgsPointXY(utm_x, utm_y)))
#
# feat['id'] = target_id
# feat['RESIDUO'] = 'recogida selectiva de textil'
# feat['UNIDADES'] = 1
# feat['HORARIO'] = 'sin restricción horaria'
# feat['CODIGO'] = 'SC017'
# feat['T1_Field2'] = 'Prolongación Ramón y Cajal, 5'
# feat['T1_Field3'] = 'https://ecoembesdudasreciclaje.es/contenedor-ropa-autorizado/'
# feat['UTMX'] = feat.geometry().asPoint().x()
# feat['UTMY'] = feat.geometry().asPoint().y()
#
# # Add the newly minted feature to the layer
# if layer.addFeature(feat):
#     layer.commitChanges()
#     print(f"[SUCCESS] Contenedor textil (id={target_id}) saved successfully!")
# else:
#     layer.rollBack()
#     print("[ERROR] Could not add the feature to the layer.")

Create QGIS project

In [1]:
import os
from qgis.core import QgsApplication, QgsProject, QgsVectorLayer

# 1. Initialize Headless QGIS
# Update the prefix path if you are using QGIS 4.0 standalone instead of OSGeo4W
QgsApplication.setPrefixPath(r"C:\OSGeo4W\apps\qgis", True)
qgs = QgsApplication([], False)
qgs.initQgis()

# 2. Define your paths
shapefile_path = r"C:\dev\QGIS-testing\Modulo3\puntos_recogida.shp"
project_path = r"C:\dev\QGIS-testing\Modulo3\ActividadModulo3.qgz"

print("[INFO] Creating new QGIS Project...")


[INFO] Creating new QGIS Project...


In [3]:
# 3. Get the project instance and clear any existing data in memory
project = QgsProject.instance()
project.clear()

# 4. Load the shapefile
layer = QgsVectorLayer(shapefile_path, "puntos_recogida", "ogr")

if not layer.isValid():
    print(f"[ERROR] Failed to load the shapefile at: {shapefile_path}")
else:
    # 5. Add the layer to the project's layer tree
    project.addMapLayer(layer)
    print(f"[SUCCESS] Layer '{layer.name()}' added to the project.")

    # 6. Save the project to disk
    project.write(project_path)
    print(f"[SUCCESS] QGIS Project saved to: {project_path}")

# 7. Cleanly shut down the QGIS engine
qgs.exitQgis()
print("[READY] You can now open ActividadModulo3.qgz in QGIS Desktop!")

[SUCCESS] Layer 'puntos_recogida' added to the project.
[SUCCESS] QGIS Project saved to: C:\dev\QGIS-testing\Modulo3\ActividadModulo3.qgz
[READY] You can now open ActividadModulo3.qgz in QGIS Desktop!


In [ ]:
# Cleanly exit the QGIS application and release all file locks/memory
qgs.exitQgis()

In [9]:
import os
import glob
import zipfile

# 1. Define the directory and base filename
base_dir = r"C:\dev\QGIS-testing\Modulo3"
base_name = "puntos_recogida"

def zip_files(base_name):
    # The path where the final ZIP will be saved
    zip_path = os.path.join(base_dir, f"{base_name}.zip")

    # 2. Find all files in that folder that start with "puntos_recogida."
    search_pattern = os.path.join(base_dir, f"{base_name}.*")
    files_to_zip = glob.glob(search_pattern)
    print(f"[INFO] Found {len(files_to_zip)} files to zip.")
    # Prevent it from trying to zip an older version of the .zip file itself
    if zip_path in files_to_zip:
        files_to_zip.remove(zip_path)

    # 3. Create the ZIP archive
    print(f"[INFO] Packaging files into {base_name}.zip...")

    # ZIP_DEFLATED applies standard compression to reduce file size
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file in files_to_zip:
            # Extract just the filename (e.g., 'puntos_recogida.shp')
            # This ensures the files are at the root of the ZIP, without the full C:\... folder structure
            file_name_only = os.path.basename(file)

            # Write it to the zip
            zipf.write(file, arcname=file_name_only)
            print(f"  -> Added: {file_name_only}")

    print(f"\n[SUCCESS] Archive saved to: {zip_path}")

zip_files("puntos_recogida")

[INFO] Found 7 files to zip.
[INFO] Packaging files into puntos_recogida.zip...
  -> Added: puntos_recogida.cpg
  -> Added: puntos_recogida.dbf
  -> Added: puntos_recogida.prj
  -> Added: puntos_recogida.qmd
  -> Added: puntos_recogida.shp
  -> Added: puntos_recogida.shx

[SUCCESS] Archive saved to: C:\dev\QGIS-testing\Modulo3\puntos_recogida.zip


In [10]:
zip_files("contenedores")

[INFO] Found 6 files to zip.
[INFO] Packaging files into contenedores.zip...
  -> Added: contenedores.cpg
  -> Added: contenedores.dbf
  -> Added: contenedores.prj
  -> Added: contenedores.shp
  -> Added: contenedores.shx

[SUCCESS] Archive saved to: C:\dev\QGIS-testing\Modulo3\contenedores.zip
